In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    lower,
    sha2,
    trim,
    when,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
dbutils.widgets.text('externalfiles_zone', 'None', '07-Externalfiles Zone')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')
externalfiles_zone = dbutils.widgets.get('externalfiles_zone')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_interaction_qca(
    spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "InteractionId	string,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	string,"
        #     "InteractionKey	string,"
        #     "InteractionCode	int,"
        #     "ExposureKey	string,"
        #     "PartyKey	int,"
        #     "CDI	string,"
        #     "InteractionTypeCode	string,"
        #     "InteractionTypeKey	string,"
        #     "InteractionType	string,"
        #     "InteractionGroupTypeCode	int,"
        #     "InteractionGroupType	string,"
        #     "InteractionCategoryCode	int,"
        #     "InteractionCategory	string,"
        #     "DialerTypeCode	string,"
        #     "DialerType	string,"
        #     "ContactTypeCode	string,"
        #     "ContactType	string,"
        #     "AttemptTypeCode	string,"
        #     "AttemptType	string,"
        #     "AttemptOutcomeCode	string,"
        #     "AttemptOutcome	string,"
        #     "AttemptSpecialMetricCode	string,"
        #     "AttemptSpecialMetric	string,"
        #     "IsPositiveFlag	boolean,"
        #     "IsPromiseFlag	boolean,"
        #     "IsActiveFlag	boolean,"
        #     "IsLegalFlag	boolean,"
        #     "IsCallFlag	boolean,"
        #     "IsFtFAppointmentFlag	boolean,"
        #     "IsRPCFlag	boolean,"
        #     "IsRPCFlagCalc	boolean,"
        #     "IsGlobalRPCFlag	boolean,"
        #     "BaseCategory	string,"
        #     "FirstLevelCategory	string,"
        #     "SecondLevelCategory	string,"
        #     "ThirdLevelCategory	string,"
        #     "NotificationType	string,"
        #     "Hierarchy	int,"
        #     "InteractionDate	timestamp,"
        #     "TaskDate	timestamp,"
        #     "InteractionComments	string,"
        #     "ResultComments	string,"
        #     "DebtorStatus	string,"
        #     "DebtorStatusComments	string,"
        #     "CurrentStrategy	string,"
        #     "CategoryOfAction	string,"
        #     "SubCategoryOfAction	string,"
        #     "Comments	string,"
        #     "CaseCode	int,"
        #     "CaseCodePrimary	string,"
        #     "ContactPerson	string,"
        #     "ContactPersonCategoryCode	int,"
        #     "ContactPersonCategory	int,"
        #     "InteractionPhone	string,"
        #     "ExternalCompanyKey	int,"
        #     "CreatedDate	timestamp,"
        #     "CreatedBy	string,"
        #     "LastUpdatedDate	date,"
        #     "LastUpdatedBy	string,"
        #     "PromiseKey	int,"
        #     "PromiseAmount	decimal(19,4),"
        #     "PromiseTypeCode	int,"
        #     "PromiseType	int,"
        #     "PromiseToPayTypeCode	int,"
        #     "PromiseToPayType	string,"
        #     "PromiseDate	timestamp,"
        #     "PromiseDueDate	string,"
        #     "PromiseStatusCode	int,"
        #     "PromiseStatus	string,"
        #     "PromiseToPayAmount	decimal(19,4),"
        #     "PromiseComments	string,"
        #     "PaymentKey	int,"
        #     "PromiseLastUpdatedDate	timestamp,"
        #     "PromiseLastUpdatedBy	string,"
        #     "CoverageOfPaymentVSPromiseAmount	decimal(19,4),"
        #     "CoverageOfPromiseVSPaymentAmount	int,"
        #     "DaysBetweenPromiseAndDueDate	int,"
        #     "ScheduleDate date,"
        #     "BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Interaction"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")
        # ================== fs_activity_leaf ==================
        fs_activity_leaf_extra_filters = None

        if fs_activity_leaf_extra_filters is not None:
            fs_activity_leaf_extra_filters.append(date_filter)
            fs_activity_leaf_final_filters = " AND ".join(
                fs_activity_leaf_extra_filters
            )
        else:
            fs_activity_leaf_final_filters = date_filter

        fs_activity_leaf_selected_fields = [
            "cac_code",
            "PD_DESC",
            "CASE_ID",
            "CAC_DIALER_PHONE",
            "EXTC_ID",
            "CAC_CR_DATE",
            "COL_ID",
            "ACTV_ID",
            "SPET_ID",
            "businessdate",
        ]

        fs_activity_leaf = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_activity_leaf",
            fs_activity_leaf_final_filters,
            fs_activity_leaf_selected_fields,
        )
        # ================== attasks ==================

        attasks_extra_filters = None

        if attasks_extra_filters is not None:
            attasks_extra_filters.append(date_filter)
            attasks_final_filters = " AND ".join(attasks_extra_filters)
        else:
            attasks_final_filters = date_filter

        attasks_selected_fields = [
            "cac_code_open",
            "task_datetime",
        ]

        attasks = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qualcoalpha_attasks",
            attasks_final_filters,
            attasks_selected_fields,
        )

        # ================== dim_cases_static ==================

        dim_cases_static_extra_filters = None

        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

        dim_cases_static_selected_fields = [
            "CASE_ACCOUNT_FOR_SEARCH",
            "CASE_code_primary",
            "cust_code",
            "CASE_CODE",
            "CSTM_TEXT_AD_20",
            "CASE_LEGAL_CONTRACT",
            "CASE_CONTRACT_ACCOUNT_KEY",
        ]

        dim_cases_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_cases_static",
            dim_cases_static_final_filters,
            dim_cases_static_selected_fields,
        )
        # ================== dim_activity_type ==================

        dim_activity_type_extra_filters = None

        if dim_activity_type_extra_filters is not None:
            dim_activity_type_extra_filters.append(date_filter)
            dim_activity_type_final_filters = " AND ".join(
                dim_activity_type_extra_filters
            )
        else:
            dim_activity_type_final_filters = date_filter

        dim_activity_type_selected_fields = [
            "ACTV_ID",
            "ACTV_INTCODE",
            "ACTV_DESC",
            "FLG_ACTIVE",
            "IS_LEGAL",
            "ACTVH1_ID",
            "ACTVH2_ID",
            "ACTVH3_ID",
        ]

        dim_activity_type = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_activity_type",
            dim_activity_type_final_filters,
            dim_activity_type_selected_fields,
        )
        # ================== dim_activity_type_hier1 ==================

        dim_activity_type_hier1_extra_filters = None

        if dim_activity_type_hier1_extra_filters is not None:
            dim_activity_type_hier1_extra_filters.append(date_filter)
            dim_activity_type_hier1_final_filters = " AND ".join(
                dim_activity_type_hier1_extra_filters
            )
        else:
            dim_activity_type_hier1_final_filters = date_filter

        dim_activity_type_hier1_selected_fields = [
            "ACTVH1_DESC",
            "ACTVH1_ID",
        ]

        dim_activity_type_hier1 = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_activity_type_hier1",
            dim_activity_type_hier1_final_filters,
            dim_activity_type_hier1_selected_fields,
        )
        # ================== dim_activity_type_hier2 ==================

        dim_activity_type_hier2_extra_filters = None

        if dim_activity_type_hier2_extra_filters is not None:
            dim_activity_type_hier2_extra_filters.append(date_filter)
            dim_activity_type_hier2_final_filters = " AND ".join(
                dim_activity_type_hier2_extra_filters
            )
        else:
            dim_activity_type_hier2_final_filters = date_filter

        dim_activity_type_hier2_selected_fields = [
            "ACTVH2_DESC",
            "ACTVH2_ID",
        ]

        dim_activity_type_hier2 = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_activity_type_hier2",
            dim_activity_type_hier2_final_filters,
            dim_activity_type_hier2_selected_fields,
        )
        dim_activity_type_hier2 = dim_activity_type_hier2.withColumn(
            "ACTVH2_DESC", col("ACTVH2_DESC").cast("string")
        )
        # ================== dim_activity_type_hier3 ==================

        dim_activity_type_hier3_extra_filters = None

        if dim_activity_type_hier3_extra_filters is not None:
            dim_activity_type_hier3_extra_filters.append(date_filter)
            dim_activity_type_hier3_final_filters = " AND ".join(
                dim_activity_type_hier3_extra_filters
            )
        else:
            dim_activity_type_hier3_final_filters = date_filter

        dim_activity_type_hier3_selected_fields = [
            "ACTVH3_DESC",
            "ACTVH3_ID",
        ]

        dim_activity_type_hier3 = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_activity_type_hier3",
            dim_activity_type_hier3_final_filters,
            dim_activity_type_hier3_selected_fields,
        )
        # ================== activities_extra_setup_ab ==================

        activities_extra_setup_ab_extra_filters = None

        if activities_extra_setup_ab_extra_filters is not None:
            activities_extra_setup_ab_extra_filters.append(date_filter)
            activities_extra_setup_ab_final_filters = " AND ".join(
                activities_extra_setup_ab_extra_filters
            )
        else:
            activities_extra_setup_ab_final_filters = date_filter

        activities_extra_setup_ab_selected_fields = [
            "ActivityType",
            "DialerActivityType",
            "PersonContactType",
            "IsPositiveRPC",
            "IsPromise",
            "IsRPC",
            "ShortDescription",
        ]

        activities_extra_setup_ab = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "params_activities_extra_setup_ab",
            activities_extra_setup_ab_final_filters,
            activities_extra_setup_ab_selected_fields,
        )

        activities_extra_setup_ab = activities_extra_setup_ab.distinct()
        # ================== activitiesmapping_ab ==================

        activitiesmapping_ab_selected_fields = [
            "BaseCategory",
            "1stlevelcategory",
            "2ndLevelCategory",
            "3rdlevelCategory",
            "NotificationsFlag",
            "hierarchy",
            "ActivityCode",
            "isrpc",
            "ActivityName",
        ]

        _raw_df = spark.read.table(f"{externalfiles_zone}.params_activitiesmapping_ab")  # type: ignore
        activitiesmapping_ab = _raw_df.select(activitiesmapping_ab_selected_fields)

        # Strip the leading '<prefix>_' from ActivityName, keeping the same column name
        activitiesmapping_ab = activitiesmapping_ab.withColumn(
            "ActivityName", regexp_replace(col("ActivityName"), "^[^_]*_", "")
        )

        # Trim trailing/leading whitespace from all string columns
        for c in activitiesmapping_ab_selected_fields:
            if c != "hierarchy":
                activitiesmapping_ab = activitiesmapping_ab.withColumn(c, trim(col(c)))

        activitiesmapping_ab = activitiesmapping_ab.distinct()
        # ================== fs_promise_leaf ==================

        fs_promise_leaf_extra_filters = None

        if fs_promise_leaf_extra_filters is not None:
            fs_promise_leaf_extra_filters.append(date_filter)
            fs_promise_leaf_final_filters = " AND ".join(fs_promise_leaf_extra_filters)
        else:
            fs_promise_leaf_final_filters = date_filter

        fs_promise_leaf_selected_fields = [
            "cupr_code",
            "FSPR_AMT_PRM",
            "PRMT_ID",
            "PD_DESC",
            "DUED_ID",
            "PRST_ID",
            "FSPR_AMT_PMT",
            "PAY_CODE",
            "CUPR_LAST_UPD",
            "CUPR_LAST_UPD_BY",
            "CUPR_PAY_VS_PROMISE_AMOUNT",
            "CUPR_PAY_COVERAGE_AMOUNT",
            "DAYD_ID",
            "CASE_ID",
            "cac_code",
        ]

        fs_promise_leaf = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_promise_leaf",
            fs_promise_leaf_final_filters,
            fs_promise_leaf_selected_fields,
        )

        # ================== dim_customer_static ==================

        dim_customer_static_extra_filters = None

        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(
                dim_customer_static_extra_filters
            )
        else:
            dim_customer_static_final_filters = date_filter

        dim_customer_static_selected_fields = [
            "cust_cif",
            "cust_id",
        ]

        dim_customer_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )
        # ================== fs_cases_snap ==================

        fs_cases_snap_extra_filters = None

        if fs_cases_snap_extra_filters is not None:
            fs_cases_snap_extra_filters.append(date_filter)
            fs_cases_snap_final_filters = " AND ".join(fs_cases_snap_extra_filters)
        else:
            fs_cases_snap_final_filters = date_filter

        fs_cases_snap_selected_fields = [
            "CASE_CODE",
            "CASE_LEGAL_PROCESS_STATUS",
            "SPET_CODE",
            "CASE_SOURCE",
        ]

        fs_cases_snap = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_cases_snap",
            fs_cases_snap_final_filters,
            fs_cases_snap_selected_fields,
        )

        # ================== case_activ ==================
 
        case_activ_extra_filters = None
 
        if case_activ_extra_filters is not None:
            case_activ_extra_filters.append(date_filter)
            case_activ_final_filters = " AND ".join(
                case_activ_extra_filters
            )
        else:
            case_activ_final_filters = date_filter
 
        case_activ_selected_fields = [
            "CAC_CODE",
            "CAC_COMMENT",
            "ACTV_CODE",
            "CASE_CODE",
            "CAC_LAST_UPD",
        ]
 
        case_activ = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_case_activ",
            case_activ_final_filters,
            case_activ_selected_fields,
        )
 
        # ================== at_activ ==================
 
        at_activ_extra_filters = None
 
        if at_activ_extra_filters is not None:
            at_activ_extra_filters.append(date_filter)
            at_activ_final_filters = " AND ".join(
                at_activ_extra_filters
            )
        else:
            at_activ_final_filters = date_filter
 
        at_activ_selected_fields = [
            "ACTV_CODE",
            "ACTV_ISPROMISE",
            "ACTV_ACTIVE",
            "ACTV_DAT",
        ]
 
        at_activ = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_activ",
            at_activ_final_filters,
            at_activ_selected_fields,
        )
 
        # ================== at_lst_of_val ==================
 
        at_lst_of_val_extra_filters = None
 
        if at_lst_of_val_extra_filters is not None:
            at_lst_of_val_extra_filters.append(date_filter)
            at_lst_of_val_final_filters = " AND ".join(
                at_lst_of_val_extra_filters
            )
        else:
            at_lst_of_val_final_filters = date_filter
 
        at_lst_of_val_selected_fields = [
            "LOV_CODE",
            "LOV_DESC",
        ]
 
        at_lst_of_val = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_lst_of_val",
            at_lst_of_val_final_filters,
            at_lst_of_val_selected_fields,
        )
        # ================== at_cust_prom ==================
 
        at_cust_prom_extra_filters = None
 
        if at_cust_prom_extra_filters is not None:
            at_cust_prom_extra_filters.append(date_filter)
            at_cust_prom_final_filters = " AND ".join(
                at_cust_prom_extra_filters
            )
        else:
            at_cust_prom_final_filters = date_filter
 
        at_cust_prom_selected_fields = [
            "CUST_CODE",
            "CASE_CODE",
            "CUPR_COMMENT",
            "cupr_amount",
            "cupr_code",
            "cupr_date",
            "cupr_due_date",
            "PAY_CODE",
            "CUPR_LAST_UPD",
            "CUPR_LAST_UPD_BY",
            "cupr_pay_type",
            "cupr_status_code",
            "cupr_pay_amount",
            "CAC_CODE",
        ]
 
        at_cust_prom = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_cust_prom",
            at_cust_prom_final_filters,
            at_cust_prom_selected_fields,
        )
        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "portfolio_id_qca"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== Joins ==================

        a = fs_activity_leaf.alias("a")
        at = attasks.alias("at")
        b = dim_cases_static.alias("b")
        cs = dim_customer_static.alias("cs")
        fs = fs_cases_snap.alias("fs")
        c = dim_activity_type.alias("c")
        d1 = dim_activity_type_hier1.alias("d1")
        d2 = dim_activity_type_hier2.alias("d2")
        d3 = dim_activity_type_hier3.alias("d3")
        e = activities_extra_setup_ab.alias("e")
        f = activitiesmapping_ab.alias("f")
        g = fs_promise_leaf.alias("g")
        j = case_activ.alias("j")
        k = at_activ.alias("k")
        lst = at_lst_of_val.alias("lst")
        prom = at_cust_prom.alias("prom")
        lstpaytype = at_lst_of_val.alias("lstpaytype")
        
        target_table_qca = (
            a.join(b, (a["CASE_ID"] == b["CASE_CODE"]), "inner")
            .join(at, (a["cac_code"] == at["cac_code_open"]), "left")
            .join(cs, (b["CUST_CODE"] == cs["CUST_ID"]), "inner")
            .join(fs, (b["CASE_CODE"] == fs["CASE_CODE"]), "left")
            .join(c, (a["ACTV_ID"] == c["ACTV_ID"]), "inner")
            .join(d1, (c["ACTVH1_ID"] == d1["ACTVH1_ID"]), "inner")
            .join(d2, (c["ACTVH2_ID"] == d2["ACTVH2_ID"]), "inner")
            .join(d3, (c["ACTVH3_ID"] == d3["ACTVH3_ID"]), "inner")
            .join(e, (c["actv_intcode"] == e["shortdescription"]), "left")
            .join(
                f,
                (c["actv_intcode"].cast("string") == f["ActivityCode"].cast("string"))
                & (c["ACTV_DESC"].cast("string") == f["ActivityName"].cast("string")),
                "left",
            )
            .join(
                g,
                (g["case_id"] == b["case_code"])
                & (a["cac_code"] == coalesce(g["cac_code"], lit(-1))),
                "left",
            )
            .join(j, (j["CAC_CODE"] == a["CAC_CODE"]), "left")
            .join(k, (k["ACTV_CODE"] == j["ACTV_CODE"]), "left")
            .join(lst, (k["ACTV_DAT"] == lst["LOV_CODE"]), "left")
            .join(prom, (a["CAC_CODE"] == prom["CAC_CODE"]), "left") 
            .join(lstpaytype, (lstpaytype["LOV_CODE"] == prom["cupr_pay_type"]), "left")        
            .join(
                portfolio,
                (portfolio["portfolio_id_qca"] == a["SPET_ID"]),
                "left",
            )
        )

        # HashKeys
        target_table_qca_InteractionID_primary_key_fields = [
            fs_activity_leaf.cac_code,
            at_cust_prom.cupr_code,
            lit("qca"),
        ]

        InteractionKey_primary_key_fields = [
            fs_activity_leaf.cac_code.cast("string"),
            coalesce(at_cust_prom["cupr_code"], lit(-1)),
        ]

        IsCallFlag_type_condition = when(
            trim(col("ACTVH2_DESC")) == lit("CONTACT"), lit("1")
        ).otherwise(lit("0"))

        IsRPC_type_condition = when(
            trim(col("ACTVH3_DESC")) == lit("Right Person Contact"), lit("1")
        ).otherwise(lit("0"))
 
        NotificationType_condition = when(
                trim(col("ACTVH3_DESC")).isin("Machine", "Letter", "SMS"),
                    col("ACTVH3_DESC")
        ).otherwise(lit(None))
 
        IsPositiveRPC_condition = when(
                trim(col("ACTVH3_DESC")).isin("Right Person Contact") &
                trim(col("ACTVH1_DESC")).isin("AGREEMENT", "COMMUNICATION", "COMPLIANCE","PROMISE","ΥΠΟΣΧΕΣΗ"),lit("1")
        ).otherwise(lit("0"))
 
        GlobalRPCFlag_condition = when(
                lower(trim(activitiesmapping_ab["isrpc"])).isin("yes"), lit("1")
        ).otherwise(lit("0"))

        target_table_qca = (
            target_table_qca.withColumn(
                "InteractionId",
                sha2(
                    concat_ws(";", *target_table_qca_InteractionID_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "InteractionKey", concat_ws(";", *InteractionKey_primary_key_fields)
            )
            .withColumn("IsCallFlag", IsCallFlag_type_condition)
            .withColumn("IsRPCFlag", IsRPC_type_condition)
            .withColumn("NotificationType", NotificationType_condition)
            .withColumn("IsPositiveRPC", IsPositiveRPC_condition)
            .withColumn("IsGlobalRPCFlag", GlobalRPCFlag_condition)
        ).distinct()

        # ================== Selection of Final Columns ==================

        result_df_qca = target_table_qca.select(
            target_table_qca.InteractionId.cast("string"),
            lit("qca").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),            
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            target_table_qca.InteractionKey,
            fs_activity_leaf.cac_code.alias("InteractionCode"),
            target_table_qca.CASE_ACCOUNT_FOR_SEARCH.alias("ExposureKey"),
            dim_cases_static.cust_code.alias("PartyKey"),
            target_table_qca.cust_cif.alias("CDI"),
            dim_activity_type.ACTV_ID.cast("string").alias("InteractionTypeCode"),
            target_table_qca.ACTV_INTCODE.cast("string").alias("InteractionTypeKey"),
            target_table_qca.ACTV_DESC.alias("InteractionType"),
            dim_activity_type_hier2.ACTVH2_ID.alias("InteractionGroupTypeCode"),
            target_table_qca.ActivityType.alias("InteractionGroupType"),
            dim_activity_type_hier1.ACTVH1_ID.cast("int").alias("InteractionCategoryCode"),
            dim_activity_type_hier1.ACTVH1_DESC.alias("InteractionCategory"),
            col("lst.LOV_CODE").cast("string").alias("DialerTypeCode"),
            col("lst.LOV_DESC").alias("DialerType"),
            dim_activity_type.ACTVH3_ID.cast("string").alias("ContactTypeCode"),
            dim_activity_type_hier3.ACTVH3_DESC.alias("ContactType"),
            dim_activity_type.ACTVH3_ID.cast("string").alias("AttemptTypeCode"),
            dim_activity_type_hier3.ACTVH3_DESC.alias("AttemptType"),
            lit(None).alias("AttemptOutcomeCode"),
            lit(None).alias("AttemptOutcome"),
            lit(None).alias("AttemptSpecialMetricCode"),
            lit(None).alias("AttemptSpecialMetric"),
            target_table_qca.IsPositiveRPC.cast("boolean").alias("IsPositiveFlag"),
            coalesce(at_activ.ACTV_ISPROMISE, target_table_qca.IsPromise.cast("boolean")).cast("boolean").alias("IsPromiseFlag"),
            at_activ.ACTV_ACTIVE.cast("boolean").alias("IsActiveFlag"),
            target_table_qca.IS_LEGAL.cast("boolean").alias("IsLegalFlag"),
            target_table_qca.IsCallFlag.cast("boolean").alias("IsCallFlag"),
            lit(None).cast("boolean").alias("IsFtFAppointmentFlag"),
            target_table_qca.IsRPCFlag.cast("boolean").alias("IsRPCFlag"),
            target_table_qca.IsRPCFlag.cast("boolean").alias("IsRPCFlagCalc"),
            target_table_qca.IsGlobalRPCFlag.cast("boolean").alias("IsGlobalRPCFlag"),
            target_table_qca.BaseCategory.alias("BaseCategory"),
            target_table_qca["1stlevelcategory"].alias("FirstLevelCategory"),
            target_table_qca["2ndLevelCategory"].alias("SecondLevelCategory"),
            target_table_qca["3rdlevelCategory"].alias("ThirdLevelCategory"),
            target_table_qca.NotificationsFlag.alias("NotificationType"),
            target_table_qca.hierarchy.cast("int").alias("Hierarchy"),
            fs_activity_leaf.PD_DESC.cast("timestamp").alias("InteractionDate"),
            attasks.task_datetime.cast("timestamp").alias("TaskDate"),
            case_activ.CAC_COMMENT.alias("InteractionComments"),
            lit(None).alias("ResultComments"),
            lit(None).alias("DebtorStatus"),
            lit(None).alias("CurrentStrategy"),
            lit(None).alias("CategoryOfAction"),
            lit(None).alias("DebtorStatusComments"),
            lit(None).alias("SubCategoryOfAction"),
            lit(None).alias("Comments"),
            fs_activity_leaf.CASE_ID.alias("CaseCode"),
            target_table_qca.CASE_code_primary.cast("string").alias("CaseCodePrimary"),
            lit(None).alias("ContactPerson"),
            lit(None).alias("ContactPersonCategoryCode"),
            lit(None).alias("ContactPersonCategory"),
            target_table_qca.CAC_DIALER_PHONE.alias("InteractionPhone"),
            target_table_qca.EXTC_ID.alias("ExternalCompanyKey"),
            target_table_qca.CAC_CR_DATE.alias("CreatedDate"),
            target_table_qca.COL_ID.cast("string").alias("CreatedBy"),
            case_activ.CAC_LAST_UPD.cast("date").alias("LastUpdatedDate"),
            lit(None).alias("LastUpdatedBy"),
            at_cust_prom.cupr_code.alias("PromiseKey"),
            at_cust_prom.cupr_amount.cast("decimal(19,4)").alias("PromiseAmount"),
            target_table_qca.PRMT_ID.alias("PromiseTypeCode"),
            target_table_qca.PRMT_ID.alias("PromiseType"),
            at_cust_prom.cupr_pay_type.alias("PromiseToPayTypeCode"),
            col("lstpaytype.LOV_DESC").alias("PromiseToPayType"),
            at_cust_prom.cupr_date.cast("timestamp").alias("PromiseDate"),
            target_table_qca.DUED_ID.cast("string").alias("PromiseDueDate"),
            target_table_qca.PRST_ID.alias("PromiseStatusCode"),
            target_table_qca.PRST_ID.cast("string").alias("PromiseStatus"),
            at_cust_prom.cupr_pay_amount.cast("decimal(19,4)").alias(
                "PromiseToPayAmount"
            ),
            at_cust_prom.CUPR_COMMENT.alias("PromiseComments"),
            at_cust_prom.PAY_CODE.alias("PaymentKey"),
            at_cust_prom.CUPR_LAST_UPD.alias("PromiseLastUpdatedDate"),
            at_cust_prom.CUPR_LAST_UPD_BY.alias("PromiseLastUpdatedBy"),
            target_table_qca.CUPR_PAY_VS_PROMISE_AMOUNT.alias(
                "CoverageOfPaymentVSPromiseAmount"
            ),
            target_table_qca.CUPR_PAY_COVERAGE_AMOUNT.alias(
                "CoverageOfPromiseVSPaymentAmount"
            ),
            target_table_qca.DAYD_ID.alias("DaysBetweenPromiseAndDueDate"),
            lit(None).alias("ScheduleDate"),
            target_table_qca.businessdate.alias("BusinessDate"),
        ).distinct()
        # ================== Append ==================
        # Disable Photon to avoid hash join skew error during write (activities_mapping fix)
        spark.conf.set("spark.databricks.photon.enabled", "false")

        result_df_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

        spark.conf.set("spark.databricks.photon.enabled", "true")
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_interaction_qca(spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix)
